# Silver Holding History
- **Purpose**: Transforms Bronze HoldingHistory into Silver by casting types and deduplicating based on the composite key, keeping the latest record.
- **Business Context**: PWG Pipeline - Trade Domain (Runs independent of dim_trade).
- **Execution Frequency**: Per Batch
- **Inputs**: `bronze.holdinghistory`
- **Outputs**: `silver.holdings` (CREATE OR REPLACE pattern)

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# Importing functions and libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Creating widgets using db-utils for reusability
dbutils.widgets.text("env_catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id", "1")

batch_id = dbutils.widgets.get("batch_id")
catalog = dbutils.widgets.get("env_catalog")

bronze_table = f"{catalog}.bronze.holdings"
silver_table = f"{catalog}.silver.holdings"

In [0]:
# Logging functions for initial load
l_df = spark.sql(f"SELECT * FROM {bronze_table} ORDER BY _ingest_ts DESC LIMIT 1")
carried_run_id = str(l_df.select("_run_id").first()[0])

log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_holdings', 'Starting processing for standalone silver holdings CDC updates')
start_pipeline_run(spark, carried_run_id, batch_id)
log_domain_run_status(spark, carried_run_id, batch_id, 'HOLDINGS', 'RUNNING')

In [0]:
# Reading bronze table in dataframe
df = spark.read.table(bronze_table)

# Reading initial count of the table
source_count = df.count()

In [0]:
# Applying required Transformations 
try:
    df2 = df.withColumns({
        "HH_H_T_ID" : col("HH_H_T_ID").cast("bigint"),
        "HH_T_ID" : col("HH_T_ID").cast("bigint"),
        "HH_BEFORE_QTY" : col("HH_BEFORE_QTY").cast("int"),
        "HH_AFTER_QTY" : col("HH_AFTER_QTY").cast("int"),
        "_load_ts": current_timestamp()
    })
except Exception as e:
    print(f"Error during transformation: {e}")

In [0]:
# Deduplication logic here
try:
    df2.createOrReplaceTempView("holding_history_view")
    df3 = spark.sql("""
        select *,
        row_number() over(
            partition by HH_H_T_ID, HH_T_ID, HH_BEFORE_QTY, HH_AFTER_QTY
            order by _ingest_ts desc
        ) as rn
        from holding_history_view
    """)
    df4 = df3.filter(col("rn") == 1).drop('rn', '_source_file', '_ingest_ts')
except Exception as e:
    print(f"Error during deduplication: {e}")
    raise e


In [0]:
# writing dataframe to silver table
df4.write.format("delta").mode("overwrite").saveAsTable(silver_table)

# Storing final count and printing it to check for any changes
target_count = df4.count()
print(f"Total count of tradehistory is {target_count}")

In [0]:
null_count = spark.sql("SELECT COUNT(*) FROM holding_history_view WHERE HH_H_T_ID IS NULL").first()[0]

log_dq_result(spark, carried_run_id, "silver.holdings", "Null HH_H_T_ID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, batch_id, 'HOLDINGS', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_holdings', 'Successfully completed standalone holdings CDC updates')

In [0]:
# Operations Logging
# Extract the carry-forwarded _run_id from dataframe
try:

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id=batch_id,
        domain="TRADE",
        table_name="holdings",
        source_layer="bronze_holdings",
        target_layer="silver",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=batch_id,
        layer="silver",
        table_name="holdings",
        operation="OVERWRITE",
        rows_affected=int(target_count)
    )

    print("Done")
except Exception as e:
    print(f"Error during operations logging: {e}")